# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [23]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "tracks_features.csv"
print(DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Unduh data ke data/raw/ lalu sesuaikan DATA_PATH.'
    )

print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

/home/jovyan/work/data/raw/tracks_features.csv
Dataset: tracks_features.csv
Ukuran file: 345.7 MB


CSV to Parquet Conversion

Mengonversi dataset Spotify dari CSV ke Parquet menggunakan DuckDB

In [24]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = PROCESSED_DIR / "tracks_features.parquet"

con = duckdb.connect()

# Baca CSV sebagai tabel sementara
con.execute(
    "CREATE OR REPLACE TEMP TABLE spotify AS SELECT * FROM read_csv_auto(?)",
    [str(DATA_PATH)]
)

# Simpan tabel sebagai Parquet
output_path = str(PARQUET_PATH).replace("'", "''")

con.execute(f"""
    COPY spotify
    TO '{output_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# Verifikasi hasil konversi
result = con.execute(
    "SELECT COUNT(*) FROM read_parquet(?)",
    [str(PARQUET_PATH)]
).fetchone()[0]

con.close()

print("Konversi berhasil!")
print(f"File: {PARQUET_PATH}")
print(f"Jumlah baris: {result:,}")
print(f"Ukuran CSV: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB")
print(f"Ukuran Parquet: {PARQUET_PATH.stat().st_size / 1_000_000:.1f} MB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Konversi berhasil!
File: /home/jovyan/work/data/processed/tracks_features.parquet
Jumlah baris: 1,204,025
Ukuran CSV: 345.7 MB
Ukuran Parquet: 74.5 MB


Update Dataset Path
Memperbarui `DATA_PATH` agar mengarah ke file Parquet yang telah dikonversi untuk digunakan pada tahap analisis selanjutnya.

In [25]:
DATA_PATH = PARQUET_PATH

## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [26]:
lazy_data = pl.scan_parquet(DATA_PATH)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

Schema({'id': String, 'name': String, 'album': String, 'album_id': String, 'artists': String, 'artist_ids': String, 'track_number': Int64, 'disc_number': Int64, 'explicit': Boolean, 'danceability': Float64, 'energy': Float64, 'key': Int64, 'loudness': Float64, 'mode': Int64, 'speechiness': Float64, 'acousticness': Float64, 'instrumentalness': Float64, 'liveness': Float64, 'valence': Float64, 'tempo': Float64, 'duration_ms': Int64, 'time_signature': Float64, 'year': Int64, 'release_date': String})


jumlah_baris,id_null,name_null,album_null,album_id_null,artists_null,artist_ids_null,track_number_null,disc_number_null,explicit_null,danceability_null,energy_null,key_null,loudness_null,mode_null,speechiness_null,acousticness_null,instrumentalness_null,liveness_null,valence_null,tempo_null,duration_ms_null,time_signature_null,year_null,release_date_null
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1204025,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [27]:
contoh_data = lazy_data.head(10).collect()
contoh_data

id,name,album,album_id,artists,artist_ids,track_number,disc_number,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,year,release_date
str,str,str,str,str,str,i64,i64,bool,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,f64,i64,str
"""7lmeHLHBe4nmXzuXc0HDjk""","""Testify""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",1,1,false,0.47,0.978,7,-5.399,1,0.0727,0.0261,0.0000109,0.356,0.503,117.906,210133,4.0,1999,"""1999-11-02"""
"""1wsRitfRRtWyEapl0q22o8""","""Guerrilla Radio""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",2,1,true,0.599,0.957,11,-5.764,1,0.188,0.0129,0.0000706,0.155,0.489,103.68,206200,4.0,1999,"""1999-11-02"""
"""1hR0fIFK2qRG3f3RF70pb7""","""Calm Like a Bomb""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",3,1,false,0.315,0.97,7,-5.424,1,0.483,0.0234,0.000002,0.122,0.37,149.749,298893,4.0,1999,"""1999-11-02"""
"""2lbASgTSoDO7MTuLAXlTW0""","""Mic Check""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",4,1,true,0.44,0.967,11,-5.83,0,0.237,0.163,0.000004,0.121,0.574,96.752,213640,4.0,1999,"""1999-11-02"""
"""1MQTmpYOZ6fcMQc56Hdo7T""","""Sleep Now In the Fire""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",5,1,false,0.426,0.929,2,-6.729,1,0.0701,0.00162,0.105,0.0789,0.539,127.059,205600,4.0,1999,"""1999-11-02"""
"""2LXPNLSMAauNJfnC58lSqY""","""Born of a Broken Man""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",6,1,false,0.298,0.848,2,-5.947,1,0.0727,0.0538,0.00152,0.201,0.194,148.282,280960,4.0,1999,"""1999-11-02"""
"""3moeHk8eIajvUEzVocXukf""","""Born As Ghosts""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",7,1,false,0.417,0.976,9,-6.032,1,0.175,0.000427,0.000134,0.107,0.483,90.395,202040,4.0,1999,"""1999-11-02"""
"""4llunZfVXv3NvUzXVB3VVL""","""Maria""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",8,1,false,0.277,0.873,11,-6.571,0,0.0883,0.00694,0.000054,0.188,0.618,172.848,228093,4.0,1999,"""1999-11-02"""
"""21Mq0NzFoVRvOmLTOnJjng""","""Voice of the Voiceless""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",9,1,false,0.441,0.882,7,-7.363,1,0.044,0.0195,0.00684,0.15,0.418,83.371,151573,4.0,1999,"""1999-11-02"""


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [28]:
connection = duckdb.connect()
profil_duckdb = connection.execute(
    'SUMMARIZE SELECT * FROM read_parquet(?)', [str(DATA_PATH)]
).pl()
profil_duckdb

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""VARCHAR""","""0001Lyv0YTjkZSqzT4WkLy""","""7zzzxEQAsiuIimY7AiTGJ9""",1118017,null,null,null,null,null,1204025,0.00
"""name""","""VARCHAR""","""!""","""﻿what Baby""",1064273,null,null,null,null,null,1204025,0.00
"""album""","""VARCHAR""","""!""","""행복(Happiness)""",129733,null,null,null,null,null,1204025,0.00
"""album_id""","""VARCHAR""","""0008WZMLnvEBVnq418uZsI""","""7zzz0Fqahwtd6jd2BgBFj6""",103327,null,null,null,null,null,1204025,0.00
"""artists""","""VARCHAR""","""[""'63 Burnout""]""","""['Ｏｃｅａｎ Ｓｈｏｒｅｓ']""",147447,null,null,null,null,null,1204025,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""tempo""","""DOUBLE""","""0.0""","""248.93400000000003""",141624,"""117.63435180997067""","""30.937046661114245""","""94.11080622186773""","""116.67323588832161""","""137.1241167263446""",1204025,0.00
"""duration_ms""","""BIGINT""","""1000""","""6061090""",217629,"""248839.86096218933""","""162210.35723030404""","""174095""","""224379""","""285932""",1204025,0.00
"""time_signature""","""DOUBLE""","""0.0""","""5.0""",5,"""3.8324943418948942""","""0.5611825532708646""","""4.0""","""4.0""","""4.0""",1204025,0.00


## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.